# Bank Marketing — Data Preprocessing

## Описание этапа

**Цель:** подготовить данные для обучения и сравнения Logistic Regression и SVM без утечки информации.

На основе результатов EDA определим способы обработки признаков, разделим данные на обучающую и тестовую выборки и построим единый пайплайн предобработки.

**Основные задачи:**
1. Исключить признаки, недоступные на момент прогнозирования.
2. Разделить данные на Train и Test.
3. Обработать специальные значения и категориальные признаки.
4. Настроить масштабирование и кодирование.
5. Собрать и проверить пайплайн предобработки.

## 1. Загрузка данных

Импорт библиотек и загрузка исходного датасета.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

# Корневая папка проекта
project_root = Path.cwd().resolve()

if project_root.name == 'notebooks':
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.preprocessing import (
    build_preprocessing_pipeline,
    transform_pdays,
    NUMERIC_COLS,
    ONEHOT_COLS,
    TARGET_COLS,
    CYCLICAL_COLS,
    BINARY_COLS
)

In [2]:
df = pd.read_csv(project_root / 'data' / 'raw' / 'banking.csv')

df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp_var_rate,cons_price_idx,cons_conf_idx,euribor3m,nr_employed,y
0,44,blue-collar,married,basic.4y,unknown,yes,no,cellular,aug,thu,...,1,999,0,nonexistent,1.4,93.444,-36.1,4.963,5228.1,0
1,53,technician,married,unknown,no,no,no,cellular,nov,fri,...,1,999,0,nonexistent,-0.1,93.200,-42.0,4.021,5195.8,0
2,28,management,single,university.degree,no,yes,no,cellular,jun,thu,...,3,6,2,success,-1.7,94.055,-39.8,0.729,4991.6,1
3,39,services,married,high.school,no,no,no,cellular,apr,fri,...,2,999,0,nonexistent,-1.8,93.075,-47.1,1.405,5099.1,0
4,55,retired,married,basic.4y,no,yes,no,cellular,aug,fri,...,1,3,1,success,-2.9,92.201,-31.4,0.869,5076.2,1


In [3]:
df.shape

(41188, 21)

## 2. Подготовка признаков

Будем прогнозировать отклик клиента перед запланированным телефонным звонком.

Исключим `duration`, поскольку продолжительность разговора становится известна только после его завершения. Остальные признаки пока сохраним, предполагая, что их значения доступны на момент прогнозирования.

Целевая переменная `y` уже закодирована значениями `0` и `1`.

In [4]:
X = df.drop(columns=['y', 'duration'])
y = df['y'].copy()

print('X:', X.shape)
print('y:', y.shape)

X: (41188, 19)
y: (41188,)


In [5]:
assert 'y' not in X.columns
assert 'duration' not in X.columns
assert len(X) == len(y)

## 3. Разделение данных

Разделим данные на обучающую (80%) и тестовую (20%) выборки с сохранением соотношения классов.

Тестовую выборку оставим для итоговой оценки моделей.

### 3.1. Train/test split

Используем стратифицированное разделение с `random_state=42` для воспроизводимости результатов.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [7]:
print('X_train:', X_train.shape)
print('X_test:', X_test.shape)
print('y_train:', y_train.shape)
print('y_test:', y_test.shape)

X_train: (32950, 19)
X_test: (8238, 19)
y_train: (32950,)
y_test: (8238,)


In [8]:
assert len(X_train) + len(X_test) == len(X)
assert len(y_train) + len(y_test) == len(y)

### 3.2. Проверка распределения классов

Проверим, сохранилось ли исходное соотношение классов после разделения.

In [9]:
class_distribution = pd.DataFrame({
    'Исходные данные': y.value_counts(normalize=True) * 100,
    'Train': y_train.value_counts(normalize=True) * 100,
    'Test': y_test.value_counts(normalize=True) * 100
})

class_distribution.round(2)

,Исходные данные,Train,Test
y,,,
0,88.73,88.73,88.74
1,11.27,11.27,11.26


## 4. Обработка особенностей данных

На основе результатов EDA определим способы обработки неизвестных и редких категорий, специальных значений и выбросов.

Проверки и настройку преобразований будем выполнять на обучающей выборке.

### 4.1. Неизвестные и редкие категории

Значения `unknown` сохраним как отдельные категории, поскольку отсутствие информации может быть полезно для классификации.

Редкие категории также оставим без изменений.

Для обработки категориальных признаков используем `OneHotEncoder` и `TargetEncoder`.

In [10]:
unknown_share = (X_train == 'unknown').mean() * 100

unknown_share[unknown_share > 0].round(2)

job           0.78
marital       0.21
education     4.22
default      20.84
housing       2.44
loan          2.44
dtype: float64

### 4.2. Обработка `pdays`

Значение `999` в признаке `pdays` обозначает отсутствие соответствующего предыдущего контакта.

Заменим исходный признак двумя переменными:

- `pdays_known` — наличие известного интервала с предыдущего контакта;
- `pdays_days` — количество дней, если интервал известен.

Для наблюдений с `pdays = 999` установим оба значения равными нулю.

In [11]:
sample = pd.DataFrame({
    'pdays': [999, 3, 10],
    'previous': [0, 1, 2]
})

transform_pdays(sample)

,previous,pdays_known,pdays_days
0,0,0,0
1,1,1,3
2,2,1,10


### Выводы

По результатам EDA выбраны следующие решения:

- `unknown` и редкие категории сохраняем без изменений.
- `pdays` преобразуем в два признака: `pdays_known` и `pdays_days`.
- Выбросы и сильно коррелирующие признаки пока сохраняем. Необходимость их обработки проверим при моделировании.

Выбранные преобразования реализуем в пайплайне предобработки.

## 5. Кодирование и масштабирование

Для подготовки данных используем несколько способов преобразования признаков:

- `OneHotEncoder` — для категориальных признаков с небольшим числом значений.
- `TargetEncoder` — для `job` и `education`, чтобы сократить размерность.
- Циклическое кодирование — для `month`, учитывая периодичность календарных месяцев.
- `StandardScaler` — для масштабирования числовых признаков.

Все преобразования будут выполняться внутри пайплайна предобработки.

### 5.1. Разделение признаков по типам

Признаки распределены по группам в зависимости от выбранного способа обработки. Их списки определены в `src/preprocessing.py`.

При этом исходный `pdays` заменяется на `pdays_days` и `pdays_known`.

In [12]:
feature_groups = {
    'Числовые': NUMERIC_COLS,
    'Target Encoding': TARGET_COLS,
    'Cyclical Encoding': CYCLICAL_COLS,
    'One-Hot Encoding': ONEHOT_COLS,
    'Бинарные': BINARY_COLS
}

for group, columns in feature_groups.items():
    print(f'{group} ({len(columns)}): {columns}')

Числовые (9): ['age', 'campaign', 'previous', 'emp_var_rate', 'cons_price_idx', 'cons_conf_idx', 'euribor3m', 'nr_employed', 'pdays_days']
Target Encoding (2): ['job', 'education']
Cyclical Encoding (1): ['month']
One-Hot Encoding (7): ['marital', 'default', 'housing', 'loan', 'contact', 'day_of_week', 'poutcome']
Бинарные (1): ['pdays_known']


### 5.2. One-Hot Encoding

Для категориальных признаков с небольшим количеством значений используем `OneHotEncoder`.

Каждая категория будет представлена отдельным бинарным признаком. Значения `unknown` и редкие категории сохранятся.

Параметр `handle_unknown='ignore'` позволит обрабатывать категории, не встречавшиеся при обучении кодировщика.

### 5.3. Target Encoding

Для `job` и `education` используем `TargetEncoder`, заменяющий категории сглаженными оценками доли положительного класса.

Чтобы снизить риск переобучения, применяем внутренний кросс-фиттинг с пятью стратифицированными фолдами.

Полученные признаки дополнительно стандартизируем.

### 5.4. Циклическое кодирование `month`

Признак `month` имеет циклическую структуру: после декабря снова наступает январь.

Преобразуем месяц в две числовые переменные с помощью синуса и косинуса. Это позволит учитывать календарную близость месяцев без создания отдельных бинарных столбцов.

Такое представление предполагает циклическую зависимость, поэтому его эффективность позднее сравним с One-Hot Encoding.

### 5.5. Масштабирование числовых признаков

Для числовых признаков используем `StandardScaler`, который вычитает среднее и делит значения на стандартное отклонение.

Параметры стандартизации будут рассчитываться только на обучающих данных внутри пайплайна.

Бинарный `pdays_known` и циклические признаки `month_sin`, `month_cos` оставим без дополнительного масштабирования.

## 6. Построение и проверка пайплайна

Объединим выбранные преобразования в единый пайплайн для дальнейшего использования с Logistic Regression и SVM.

### 6.1. Создание пайплайна

Используем функцию `build_preprocessing_pipeline()` из `src.preprocessing`.

Сначала выполняется обработка `pdays`, затем `ColumnTransformer` применяет кодирование и масштабирование к соответствующим группам признаков.

In [13]:
preprocessing_pipeline = build_preprocessing_pipeline()

preprocessing_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('pdays', ...), ('columns', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"func func: callable, default=NoneThe callable to use for the transformation. This will be passedthe same arguments as transform, with args and kwargs forwarded.If func is None, then func will be the identity function.",<function tra...t 0x118da5760>
,"inverse_func inverse_func: callable, default=NoneThe callable to use for the inverse transformation. This will bepassed the same arguments as inverse transform, with args andkwargs forwarded. If inverse_func is None, then inverse_funcwill be the identity function.",None
,"validate validate: bool, default=FalseIndicate that the input X array should be checked before calling``func``. The possibilities are:- If False, there is no input validation.- If True, then X will be converted to a 2-dimensional NumPy array or sparse matrix. If the conversion is not possible an exception is raised... versionchanged:: 0.22 The default of ``validate`` changed from True to False.",False
,"accept_sparse accept_sparse: bool, default=FalseIndicate that func accepts a sparse matrix as input. If validate isFalse, this has no effect. Otherwise, if accept_sparse is false,sparse matrix inputs will cause an exception to be raised.",False
,"check_inverse check_inverse: bool, default=TrueWhether to check that or ``func`` followed by ``inverse_func`` leads tothe original inputs. It can be used for a sanity check, raising awarning when the condition is not fulfilled... versionadded:: 0.20",True
,"feature_names_out feature_names_out: callable, 'one-to-one' or None, default=NoneDetermines the list of feature names that will be returned by the`get_feature_names_out` method. If it is 'one-to-one', then the outputfeature names will be equal to the input feature names. If it is acallable, then it must take two positional arguments: this`FunctionTransformer` (`self`) and an array-like of input feature names(`input_features`). It must return an array-like of output featurenames. The `get_feature_names_out` method is only defined if`feature_names_out` is not None.See ``get_feature_names_out`` for more details... versionadded:: 1.1",None
,"kw_args kw_args: dict, default=NoneDictionary of ad

### 6.2. Проверка работы пайплайна

Применим пайплайн к обучающей выборке и проверим размерность полученной матрицы, названия признаков и наличие некорректных значений.

Тестовую выборку на данном этапе не используем.

In [14]:
X_train_transformed = preprocessing_pipeline.fit_transform(
    X_train,
    y_train
)

In [15]:
print('До предобработки:', X_train.shape)
print('После предобработки:', X_train_transformed.shape)

До предобработки: (32950, 19)
После предобработки: (32950, 37)


In [16]:
feature_names = (
    preprocessing_pipeline
    .named_steps['columns']
    .get_feature_names_out()
)

X_train_prepared = pd.DataFrame(
    X_train_transformed,
    columns=feature_names,
    index=X_train.index
)

X_train_prepared.head()

,numeric__age,numeric__campaign,numeric__previous,numeric__emp_var_rate,numeric__cons_price_idx,numeric__cons_conf_idx,numeric__euribor3m,numeric__nr_employed,numeric__pdays_days,onehot__marital_divorced,...,onehot__day_of_week_tue,onehot__day_of_week_wed,onehot__poutcome_failure,onehot__poutcome_nonexistent,onehot__poutcome_success,target__job,target__education,month__month_sin,month__month_cos,binary__pdays_known
27549,-0.964771,0.891970,-0.350073,-0.116686,-0.649341,-0.325891,0.286949,0.397226,-0.165648,0.0,...,0.0,1.0,0.0,1.0,0.0,-0.022073,1.112328,-0.500000,0.866025,0.0
27965,-0.197393,-0.202563,-0.350073,0.646775,0.722568,0.883719,0.712262,0.330917,-0.165648,0.0,...,0.0,0.0,0.0,1.0,0.0,0.387508,1.143506,0.500000,-0.866025,0.0
36497,1.049596,-0.567408,1.665464,-1.198255,-1.179790,-1.233098,-1.349182,-0.938621,-0.165648,0.0,...,0.0,1.0,1.0,0.0,0.0,-0.841948,-0.181111,0.500000,-0.866025,0.0
315,-0.293315,-0.567408,-0.350073,0.837640,-0.227747,0.948520,0.772774,0.843429,-0.165648,0.0,...,0.0,0.0,0.0,1.0,0.0,-0.090905,-0.032864,-0.866025,-0.500000,0.0
38191,0.282218,-0.567408,-0.350073,-0.116686,-0.649341,-0.325891,0.305967,0.397226,-0.165648,0.0,...,1.0,0.0,0.0,1.0,0.0,-0.052761,1.143506,-0.500000,0.866025,0.0


In [17]:
print('Размерность:', X_train_prepared.shape)

print(
    'Пропущенные значения:',
    X_train_prepared.isna().sum().sum()
)

print(
    'Бесконечные значения:',
    np.isinf(X_train_prepared.to_numpy()).sum()
)

Размерность: (32950, 37)
Пропущенные значения: 0
Бесконечные значения: 0


In [18]:
assert len(X_train_prepared) == len(X_train)
assert X_train_prepared.columns.is_unique
assert np.isfinite(X_train_prepared.to_numpy()).all()
assert X_train_prepared.index.equals(X_train.index)

## 7. Итоги предобработки

В ходе предобработки подготовлен единый пайплайн для обучения Logistic Regression и SVM.

**Основные результаты:**

- Исключён `duration`, недоступный на момент прогнозирования.
- Данные разделены на Train (80%) и Test (20%) со стратификацией.
- `pdays` преобразован в `pdays_known` и `pdays_days`.
- Для категориальных признаков использованы One-Hot Encoding, Target Encoding с кросс-фиттингом и циклическое кодирование.
- Числовые признаки стандартизированы. Выбросы, редкие категории и коррелирующие признаки сохранены.

Предобработка вынесена в `src/preprocessing.py` и объединена в единый `Pipeline`.

**Результат:** обучающая выборка преобразована в матрицу размером 32 950 × 37. Пропущенные и бесконечные значения отсутствуют.

### Следующий этап

В `03_modeling.ipynb` обучим Logistic Regression, Linear SVM и RBF SVM с использованием кросс-валидации.

Дополнительно сравним комбинированное кодирование с базовым One-Hot Encoding по ROC-AUC и Average Precision (PR-AUC).